# Moduł 13: Sieci rekurencyjne (RNN) i LSTM

RNN to sieci zaprojektowane do przetwarzania **danych sekwencyjnych** (tekst, serie czasowe, mowa).

## Spis treści
1. Problem danych sekwencyjnych
2. Vanilla RNN
3. Problem zanikającego/eksplodującego gradientu
4. LSTM (Long Short-Term Memory)
5. RNN/LSTM w PyTorch
6. Perplexity
7. **Zadania**

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

## 1. Problem danych sekwencyjnych

MLP i CNN traktują wejście jako **stałej wielkości** wektor. Ale dane sekwencyjne:
- Mają **zmienną długość** — zdania, serie czasowe, dźwięk
- **Kolejność** ma znaczenie: "pies goni kota" ≠ "kot goni psa"
- Wymagają **pamięci** kontekstu — znaczenie słowa zależy od poprzednich

### Formalizacja

Dane sekwencyjne to ciąg $\mathbf{x} = (x_1, x_2, \ldots, x_T)$ o zmiennej długości $T$.

Model sekwencyjny to funkcja: $P(y | x_1, x_2, \ldots, x_T)$

### Zastosowania RNN

| Architektura | Wejście → Wyjście | Przykład |
|-------------|-------------------|---------|
| **Many-to-one** | Sekwencja → 1 wartość | Analiza sentymentu |
| **One-to-many** | 1 wartość → sekwencja | Generowanie tekstu z obrazu |
| **Many-to-many** | Sekwencja → sekwencja | Tłumaczenie, NER |
| **Seq2Seq** | Sekwencja → sekwencja (inna długość) | Tłumaczenie maszynowe |

### Modelowanie języka (Language Modeling)

Cel: modelować $P(w_1, w_2, \ldots, w_T)$ — rozkład prawdopodobieństwa nad sekwencjami słów.

Z reguły łańcuchowej:

$$P(w_1, \ldots, w_T) = \prod_{t=1}^{T} P(w_t | w_1, \ldots, w_{t-1})$$

RNN aproksymuje $P(w_t | w_1, \ldots, w_{t-1})$ za pomocą **ukrytego stanu** $h_t$ jako skompresowanej historii.

## 2. Vanilla RNN

RNN przetwarza sekwencję **krok po kroku**, utrzymując **ukryty stan** (hidden state) $h_t$ jako "pamięć".

### Wzory (dla każdego kroku $t$)

$$h_t = \tanh(W_{xh} \cdot x_t + W_{hh} \cdot h_{t-1} + b_h)$$

$$y_t = W_{hy} \cdot h_t + b_y$$

Gdzie:
- $x_t \in \mathbb{R}^d$ — wejście w kroku $t$ (np. embedding słowa)
- $h_t \in \mathbb{R}^n$ — ukryty stan w kroku $t$ (pamięć)
- $h_{t-1} \in \mathbb{R}^n$ — poprzedni ukryty stan, $h_0 = \mathbf{0}$
- $y_t \in \mathbb{R}^m$ — wyjście (logity)
- $W_{xh} \in \mathbb{R}^{n \times d}$, $W_{hh} \in \mathbb{R}^{n \times n}$, $W_{hy} \in \mathbb{R}^{m \times n}$ — macierze wag

### Równoważna notacja (skonkatenowana)

$$h_t = \tanh\left(W \begin{bmatrix} h_{t-1} \\ x_t \end{bmatrix} + b\right)$$

gdzie $W = [W_{hh} | W_{xh}] \in \mathbb{R}^{n \times (n+d)}$

**Kluczowe:** $W_{xh}$, $W_{hh}$, $W_{hy}$ są **współdzielone** w każdym kroku!

### Liczba parametrów

$$\text{params} = n(d + n + 1) + m(n + 1)$$

Niezależna od długości sekwencji $T$!

### Schemat przetwarzania (unrolling)

```
x₁ → [RNN] → h₁ → y₁
 ↓
x₂ → [RNN] → h₂ → y₂
 ↓
x₃ → [RNN] → h₃ → y₃
```

### Bidirectional RNN

Dwa RNN: jedno czyta sekwencję od lewej do prawej $\overrightarrow{h_t}$, drugie od prawej do lewej $\overleftarrow{h_t}$:

$$h_t = [\overrightarrow{h_t}; \overleftarrow{h_t}]$$

Każde słowo "widzi" kontekst z obu stron. Wymiar hidden state: $2n$.

In [ ]:
# Vanilla RNN — implementacja od zera (NumPy)
class SimpleRNN:
 def __init__(self, input_size, hidden_size, output_size):
 # Losowa inicjalizacja wag
 scale = 0.01
 self.Wxh = np.random.randn(hidden_size, input_size) * scale
 self.Whh = np.random.randn(hidden_size, hidden_size) * scale
 self.Why = np.random.randn(output_size, hidden_size) * scale
 self.bh = np.zeros((hidden_size, 1))
 self.by = np.zeros((output_size, 1))
 self.hidden_size = hidden_size

 def forward(self, inputs):
 """inputs: lista wejść [x1, x2, ..., xT]"""
 h = np.zeros((self.hidden_size, 1)) # początkowy ukryty stan
 outputs = []
 hiddens = [h]

 for x in inputs:
 x = x.reshape(-1, 1) # kolumna
 h = np.tanh(self.Wxh @ x + self.Whh @ h + self.bh)
 y = self.Why @ h + self.by
 outputs.append(y)
 hiddens.append(h)

 return outputs, hiddens

# Test — sekwencja 5 kroków, wejście 3D, ukryty stan 4D, wyjście 2D
rnn = SimpleRNN(input_size=3, hidden_size=4, output_size=2)
inputs = [np.random.randn(3) for _ in range(5)]
outputs, hiddens = rnn.forward(inputs)

print(f"Liczba kroków: {len(outputs)}")
print(f"Kształt wyjścia: {outputs[0].shape}")
print(f"Ukryty stan [0]: {hiddens[0].flatten()}")
print(f"Ukryty stan [5]: {hiddens[5].flatten()}")

## 3. Problem zanikającego / eksplodującego gradientu

### Backpropagation Through Time (BPTT)

RNN trenujemy "rozwijając" sieć w czasie i stosując backpropagation. Gradient z kroku $T$ do kroku $k$:

$$\frac{\partial L_T}{\partial h_k} = \frac{\partial L_T}{\partial h_T} \prod_{t=k+1}^{T} \frac{\partial h_t}{\partial h_{t-1}}$$

Ponieważ $h_t = \tanh(W_{hh} h_{t-1} + W_{xh} x_t + b)$:

$$\frac{\partial h_t}{\partial h_{t-1}} = W_{hh}^T \cdot \text{diag}(1 - h_t^2)$$

Zatem gradient jest mnożony przez $W_{hh}$ w **każdym kroku**:

$$\frac{\partial L}{\partial h_0} \propto \prod_{t=1}^{T} W_{hh}^T \cdot \text{diag}(1 - h_t^2)$$

### Konsekwencje

Niech $\lambda_{\max}$ to dominująca wartość własna $W_{hh}$:
- Jeśli $|\lambda_{\max}| < 1$ → gradient **zanika** eksponencjalnie: $\sim |\lambda_{\max}|^T \to 0$
 - Sieć nie uczy się **długich** zależności (np. podmiot 20 słów wcześniej)
- Jeśli $|\lambda_{\max}| > 1$ → gradient **eksploduje**: $\sim |\lambda_{\max}|^T \to \infty$
 - Wagi skaczą do nieskończoności, trening jest niestabilny

### Rozwiązania

**Gradient clipping** (przeciw eksplozji):
$$\mathbf{g} \leftarrow \frac{\mathbf{g}}{\|\mathbf{g}\|} \cdot \text{threshold} \quad \text{jeśli } \|\mathbf{g}\| > \text{threshold}$$

**Truncated BPTT** — propaguj gradient tylko przez $k$ ostatnich kroków.

**Lepsze architektury:** LSTM, GRU — zaprojektowane tak, by gradient mógł przepływać bez zmian.

In [ ]:
# Demonstracja zanikającego gradientu
W = 0.9 # wartość własna < 1
grad_products = [W ** t for t in range(50)]

W2 = 1.1 # wartość własna > 1
grad_products2 = [W2 ** t for t in range(50)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(grad_products)
axes[0].set_title("Zanikający gradient (|W| = 0.9)")
axes[0].set_xlabel("Krok czasowy")
axes[0].set_ylabel("Siła gradientu")

axes[1].plot(grad_products2)
axes[1].set_title("Eksplodujący gradient (|W| = 1.1)")
axes[1].set_xlabel("Krok czasowy")
axes[1].set_ylabel("Siła gradientu")

plt.tight_layout()
plt.show()

## 4. LSTM (Long Short-Term Memory, Hochreiter & Schmidhuber, 1997)

LSTM rozwiązuje problem zanikającego gradientu przez **bramki (gates)** — mechanizmy kontrolujące przepływ informacji.

### Architektura LSTM

Obok ukrytego stanu $h_t \in \mathbb{R}^n$, LSTM utrzymuje **stan komórki (cell state)** $c_t \in \mathbb{R}^n$ — "autostrady" informacji.

### 3 bramki (gates) — wszystkie mają postać $\sigma(W \cdot [\ldots] + b) \in (0, 1)^n$

**1. Forget gate** — jaką frakcję pamięci zachować:
$$f_t = \sigma(W_f \cdot [h_{t-1}, x_t] + b_f)$$

**2. Input gate** — jaką nową informację dodać:
$$i_t = \sigma(W_i \cdot [h_{t-1}, x_t] + b_i)$$
$$\tilde{c}_t = \tanh(W_c \cdot [h_{t-1}, x_t] + b_c) \quad \text{(kandydat na nowy stan)}$$

**3. Output gate** — co wypuścić jako ukryty stan:
$$o_t = \sigma(W_o \cdot [h_{t-1}, x_t] + b_o)$$

### Aktualizacja stanu komórki i ukrytego stanu

$$c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t$$

$$h_t = o_t \odot \tanh(c_t)$$

gdzie $\odot$ to iloczyn element-wise (Hadamard product).

### Dlaczego LSTM rozwiązuje zanikający gradient?

Gradient przepływa przez cell state $c_t$ niemal bez zmian:

$$\frac{\partial c_t}{\partial c_{t-1}} = f_t$$

Jeśli $f_t \approx 1$ (forget gate otwarty), gradient przepływa **bez zaniku**! W Vanilla RNN gradient musi przejść przez $\tanh$ i mnożenie przez $W_{hh}$ — w LSTM idzie "autostradą" $c_t$.

### Liczba parametrów LSTM

4 zestawy bramek, każdy z macierzami $W \in \mathbb{R}^{n \times (n + d)}$ i biasem $b \in \mathbb{R}^n$:

$$\text{params} = 4 \cdot [n(n + d) + n] = 4n(n + d + 1)$$

→ LSTM ma **4× więcej parametrów** niż Vanilla RNN o tym samym hidden size.

### GRU (Gated Recurrent Unit, Cho et al., 2014)

Uproszczona wersja LSTM — łączy forget i input gate w jeden **update gate**:

$$z_t = \sigma(W_z \cdot [h_{t-1}, x_t]) \quad \text{(update gate)}$$
$$r_t = \sigma(W_r \cdot [h_{t-1}, x_t]) \quad \text{(reset gate)}$$
$$\tilde{h}_t = \tanh(W \cdot [r_t \odot h_{t-1}, x_t])$$
$$h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t$$

GRU ma **3 zestawy wag** (vs 4 w LSTM) → mniej parametrów, porównywalna jakość.

| Cecha | LSTM | GRU |
|-------|------|-----|
| Bramki | 3 (forget, input, output) | 2 (update, reset) |
| Stan komórki | Tak ($c_t$ + $h_t$) | Nie (tylko $h_t$) |
| Parametry | $4n(n+d+1)$ | $3n(n+d+1)$ |
| Wydajność | Zazwyczaj lepsza dla długich sekwencji | Szybsza, porównywalna jakość |

## 5. RNN / LSTM w PyTorch

In [ ]:
# PyTorch RNN vs LSTM
batch_size = 4
seq_len = 10
input_size = 8
hidden_size = 16

# Losowe dane: (batch, seq_len, input_size)
x = torch.randn(batch_size, seq_len, input_size)

# RNN
rnn = nn.RNN(input_size, hidden_size, batch_first=True)
output_rnn, h_n_rnn = rnn(x)
print(f"RNN output: {output_rnn.shape}") # (4, 10, 16) — output na każdym kroku
print(f"RNN h_n: {h_n_rnn.shape}") # (1, 4, 16) — ostatni ukryty stan

# LSTM
lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
output_lstm, (h_n_lstm, c_n_lstm) = lstm(x)
print(f"\nLSTM output: {output_lstm.shape}")
print(f"LSTM h_n: {h_n_lstm.shape}")
print(f"LSTM c_n: {c_n_lstm.shape}") # cell state!

In [ ]:
# Przykład: przewidywanie następnego znaku (character-level language model)

# Prosty korpus
text = "hello world hello pytorch hello deep learning"
chars = sorted(set(text))
char2idx = {c: i for i, c in enumerate(chars)}
idx2char = {i: c for c, i in char2idx.items()}
vocab_size = len(chars)

print(f"Vocabulary ({vocab_size}): {chars}")

# Model LSTM do generowania tekstu
class CharLSTM(nn.Module):
 def __init__(self, vocab_size, embed_dim, hidden_dim):
 super().__init__()
 self.embedding = nn.Embedding(vocab_size, embed_dim)
 self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
 self.fc = nn.Linear(hidden_dim, vocab_size)

 def forward(self, x, hidden=None):
 x = self.embedding(x)
 output, hidden = self.lstm(x, hidden)
 logits = self.fc(output)
 return logits, hidden

model = CharLSTM(vocab_size, embed_dim=16, hidden_dim=64)
print(model)

In [ ]:
# Trening character-level modelu
# Przygotowanie danych: wejście = tekst[:-1], cel = tekst[1:]
encoded = [char2idx[c] for c in text]
input_seq = torch.tensor(encoded[:-1]).unsqueeze(0) # (1, n-1)
target_seq = torch.tensor(encoded[1:]).unsqueeze(0) # (1, n-1)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

losses = []
for epoch in range(500):
 optimizer.zero_grad()
 logits, _ = model(input_seq)
 loss = criterion(logits.view(-1, vocab_size), target_seq.view(-1))
 loss.backward()
 torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0) # gradient clipping!
 optimizer.step()
 losses.append(loss.item())
 if (epoch + 1) % 100 == 0:
 print(f"Epoch {epoch+1}: loss = {loss.item():.4f}")

plt.plot(losses)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Character LSTM — krzywa uczenia")
plt.show()

In [ ]:
# Generowanie tekstu!
def generate(model, start_char, length=50):
 model.eval()
 chars_generated = [start_char]
 x = torch.tensor([[char2idx[start_char]]])
 hidden = None

 with torch.no_grad():
 for _ in range(length):
 logits, hidden = model(x, hidden)
 probs = torch.softmax(logits[0, -1], dim=0)
 idx = torch.multinomial(probs, 1).item()
 chars_generated.append(idx2char[idx])
 x = torch.tensor([[idx]])

 return ''.join(chars_generated)

print("Generated:", generate(model, 'h', 50))

## 6. Perplexity — metryka modeli językowych

**Perplexity (PPL)** mierzy jak "zaskoczony" jest model następnym tokenem:

$$\text{PPL} = \exp\left(\frac{1}{N} \sum_{i=1}^{N} -\log P(w_i | w_1, \ldots, w_{i-1})\right) = \exp(\mathcal{L}_{\text{CE}})$$

### Interpretacja

PPL odpowiada **średniej liczbie słów**, jakie model "rozważa" na każdym kroku:
- $\text{PPL} = 1$ → model jest pewny (idealny)
- $\text{PPL} = 10$ → model wybiera spośród ~10 kandydatów
- $\text{PPL} = |V|$ → model zgaduje losowo (brak wiedzy)

### Związek z entropią

$$\text{PPL} = 2^{H(P, Q)}$$

gdzie $H(P, Q) = -\frac{1}{N}\sum_i \log_2 P(w_i)$ to cross-entropy w bitach.

### Porównanie modeli

| Model | PPL (WikiText-2) |
|-------|-----------------|
| n-gram (Kneser-Ney) | ~140 |
| LSTM (1 warstwa) | ~80-100 |
| LSTM (2 warstwy) | ~60-70 |
| Transformer (GPT-2 small) | ~29 |
| GPT-3 (175B) | ~20 |

> **Uwaga:** PPL jest porównywalny **tylko** między modelami na **tym samym** zbiorze testowym z **tym samym** tokenizerem!

---
---
## Zadania do samodzielnego rozwiązania

---

### Zadanie 1: Analiza sentymentu z LSTM (trudne)

Zbuduj prosty klasyfikator sentymentu:
1. Stwórz mały zbiór danych (20 zdań: 10 pozytywnych, 10 negatywnych)
2. Tokenizuj, zbuduj słownik, zamień na indeksy
3. Padding do jednakowej długości
4. Model: Embedding → LSTM → Linear(hidden, 2)
5. Użyj **ostatniego ukrytego stanu** do klasyfikacji
6. Trenuj i oceń accuracy

In [ ]:
# Zadanie 1
# TWÓJ KOD TUTAJ

### Zadanie 2: Prognozowanie sinusoidy (średnie)

Użyj LSTM do prognozowania wartości sinusoidy:
1. Generuj dane: sin(x) z szumem
2. Wejście: okna 20 kroków → Predykcja: następny krok
3. Trenuj LSTM
4. Wizualizuj: prawdziwa sinusoida vs predykcje

In [ ]:
# Zadanie 2: Sinusoida LSTM
np.random.seed(42)
# TWÓJ KOD TUTAJ

### Zadanie 3: Bidirectional LSTM (średnie)

W bidirectional LSTM informacja płynie w **obu kierunkach** (przód → tył i tył → przód).

1. Zmodyfikuj model z Zadania 1 na **bidirectional LSTM**
2. Podpowiedź: `nn.LSTM(..., bidirectional=True)` → hidden_size × 2
3. Porównaj wynik z unidirectional

In [ ]:
# Zadanie 3: Bidirectional LSTM
# TWÓJ KOD TUTAJ

### Zadanie 4: GRU vs LSTM porownanie (srednie)

Na jednym zbiorze danych (np. prognozowanie sinusoidy z szumem):
1. Wytrenuj model z LSTM
2. Wytrenuj identyczny model z GRU (ta sama liczba parametrow)
3. Porownaj:
 - Czas treningu
 - Koncowa wartosc loss
 - Jakosc predykcji (wizualnie)
4. Kiedy GRU jest lepsza niz LSTM?

In [ ]:
# Zadanie 4: GRU vs LSTM
import torch
import torch.nn as nn
import numpy as np
import time
torch.manual_seed(42)
# TWOJ KOD TUTAJ

### Zadanie 5: Teacher forcing (trudne)

Zaimplementuj trening seq2seq z **teacher forcing ratio**:

1. Zadanie: generowanie sekwencji (np. odwracanie listy cyfr)
2. Encoder: LSTM przetwarzajacy wejscie
3. Decoder: LSTM generujacy wyjscie krok po kroku
4. Teacher forcing ratio = 0.5: w 50% krokow podaj prawdziwy output, w 50% podaj predykcje modelu
5. Porownaj trening z ratio 0.0, 0.5, 1.0 — jak wplywa na zbieznosc?

In [ ]:
# Zadanie 5: Teacher forcing
import torch
import torch.nn as nn
import numpy as np
torch.manual_seed(42)
# TWOJ KOD TUTAJ

### Zadanie 6: Packed sequences i zmienna dlugosc (srednie)

W praktyce sekwencje maja rozne dlugosci. PyTorch obsluguje to za pomoca `pack_padded_sequence`.

1. Stworz batch 5 sekwencji o dlugosciach [3, 7, 5, 2, 6]
2. Zastosuj padding do max dlugosci
3. Uzyj `pack_padded_sequence` i `pad_packed_sequence`
4. Przepusc przez LSTM
5. Pokaz, ze wynik jest **identyczny** jak przy przetwarzaniu kazdej sekwencji osobno

In [ ]:
# Zadanie 6: Packed sequences
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
torch.manual_seed(42)
# TWOJ KOD TUTAJ

### Zadanie 7: Predykcja szeregu czasowego z LSTM
Wygeneruj syntetyczny szereg czasowy: $y = \sin(t) + 0.5\sin(3t) + \epsilon$.
1. Przygotuj dane: okno wejściowe (20 kroków) -> predykcja (1 krok)
2. Wytrenuj LSTM (1 warstwa, hidden=32)
3. Narysuj: prawdziwe wartości vs predykcje na zbiorze testowym
4. Oblicz MSE i MAE

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

# TWOJ KOD TUTAJ


### Zadanie 8: Porównanie RNN vs LSTM vs GRU
Na zadaniu klasyfikacji sentymentu (np. prosty zbiór recenzji):
1. Zaimplementuj 3 modele: Vanilla RNN, LSTM, GRU (identyczna architektura poza typem komórki)
2. Porównaj: accuracy, czas treningu, stabilność gradientów
3. Narysuj normy gradientów w trakcie treningu dla każdego modelu
4. Który model jest najstabilniejszy?

In [ ]:
import torch
import torch.nn as nn

# TWOJ KOD TUTAJ


### Zadanie 9: Bidirectional LSTM
Zaimplementuj BiLSTM dla klasyfikacji sekwencji:
1. Wygeneruj syntetyczne dane: sekwencje z wzorcami (palindromy, rosnące, losowe)
2. Porównaj LSTM vs BiLSTM na tym zadaniu
3. Dlaczego BiLSTM powinien być lepszy na palindromach?

In [ ]:
import torch
import torch.nn as nn
import numpy as np

# TWOJ KOD TUTAJ


---

## Dodatek OAI: Ćwiczenia w stylu olimpiadowym

Architektury sekwencyjne pojawiają się w zadaniach NLP i analizy szeregów czasowych.

### Powiązane zadania OAI:
- **I OAI finał**: Tłumaczenie maszynowe (Bahdanau attention — encoder-decoder)
- **II OAI**: Wykrywanie anomalii w EKG (szeregi czasowe)
- **II OAI**: Ukryte podciągi (pattern matching w sekwencjach)
- **III OAI**: Klasyfikacja audio (dane sekwencyjne)

### Ćwiczenia:
- **OAI-11A**: Detekcja anomalii w szeregu czasowym (LSTM Autoencoder)
- **OAI-11B**: Seq2seq z atencją Bahdanau (prosty translator)
- **OAI-11C**: Klasyfikacja sekwencji z Bidirectional LSTM

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

# === OAI-11A: Detekcja anomalii LSTM Autoencoder ===
print("=== LSTM Autoencoder — Detekcja anomalii (wzorzec: II OAI EKG) ===")

class LSTMAnomalyDetector(nn.Module):
 """LSTM Autoencoder do wykrywania anomalii w szeregach czasowych."""
 def __init__(self, input_dim=1, hidden_dim=32, n_layers=1):
 super().__init__()
 self.encoder = nn.LSTM(input_dim, hidden_dim, n_layers, batch_first=True)
 self.decoder = nn.LSTM(hidden_dim, hidden_dim, n_layers, batch_first=True)
 self.output_layer = nn.Linear(hidden_dim, input_dim)

 def forward(self, x):
 # Encode
 _, (h, c) = self.encoder(x)
 # Decode — powtórz hidden state jako input
 seq_len = x.size(1)
 decoder_input = h[-1].unsqueeze(1).repeat(1, seq_len, 1)
 decoder_out, _ = self.decoder(decoder_input)
 return self.output_layer(decoder_out)

# Generujemy dane: normalne sinusoidy + anomalie
np.random.seed(42)
t = np.linspace(0, 4 * np.pi, 100)
normal_signal = np.sin(t) + 0.1 * np.random.randn(len(t))
anomaly_signal = normal_signal.copy()
anomaly_signal[40:50] += 3.0 # Anomalia: nagły skok

# Trenujemy na normalnych danych
model_ae = LSTMAnomalyDetector(input_dim=1, hidden_dim=16)
optimizer = torch.optim.Adam(model_ae.parameters(), lr=1e-3)
criterion = nn.MSELoss()

# Przygotuj dane treningowe (okna czasowe)
window_size = 20
X_train = []
for i in range(0, len(normal_signal) - window_size):
 X_train.append(normal_signal[i:i+window_size])
X_train = torch.FloatTensor(np.array(X_train)).unsqueeze(-1)

# Trening
model_ae.train()
for epoch in range(30):
 optimizer.zero_grad()
 recon = model_ae(X_train)
 loss = criterion(recon, X_train)
 loss.backward()
 optimizer.step()
print(f" Trening zakończony. Loss: {loss.item():.6f}")

# Detekcja anomalii
model_ae.eval()
X_test = []
for i in range(0, len(anomaly_signal) - window_size):
 X_test.append(anomaly_signal[i:i+window_size])
X_test = torch.FloatTensor(np.array(X_test)).unsqueeze(-1)

with torch.no_grad():
 recon_test = model_ae(X_test)
 errors = ((X_test - recon_test) ** 2).mean(dim=(1, 2)).numpy()

# Wizualizacja
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6))
ax1.plot(anomaly_signal, 'b-', label='Sygnał z anomalią')
ax1.axvspan(40, 50, alpha=0.3, color='red', label='Anomalia')
ax1.set_title("Sygnał wejściowy")
ax1.legend()

ax2.plot(errors, 'r-', linewidth=2)
threshold = np.percentile(errors, 95)
ax2.axhline(y=threshold, color='k', linestyle='--', label=f'Próg = {threshold:.4f}')
ax2.set_title("Błąd rekonstrukcji (wysoki = anomalia)")
ax2.set_xlabel("Pozycja okna")
ax2.legend()
plt.tight_layout()
plt.show()

print(f" Próg anomalii: {threshold:.4f}")
print(f" Wykryte anomalie: {(errors > threshold).sum()} okien")

# === OAI-11B: Prosty Seq2Seq z atencją ===
print("\n=== Seq2Seq z atencją Bahdanau (wzorzec: I OAI finał) ===")

class BahdanauAttention(nn.Module):
 """Atencja Bahdanau — kluczowy mechanizm z I OAI."""
 def __init__(self, hidden_dim):
 super().__init__()
 self.W1 = nn.Linear(hidden_dim, hidden_dim)
 self.W2 = nn.Linear(hidden_dim, hidden_dim)
 self.V = nn.Linear(hidden_dim, 1)

 def forward(self, decoder_hidden, encoder_outputs):
 # decoder_hidden: (batch, hidden)
 # encoder_outputs: (batch, seq_len, hidden)
 decoder_hidden = decoder_hidden.unsqueeze(1) # (batch, 1, hidden)
 scores = self.V(torch.tanh(self.W1(encoder_outputs) + self.W2(decoder_hidden)))
 weights = torch.softmax(scores, dim=1) # (batch, seq_len, 1)
 context = (weights * encoder_outputs).sum(dim=1) # (batch, hidden)
 return context, weights.squeeze(-1)

# Demo
batch, seq_len, hidden = 2, 5, 8
attn = BahdanauAttention(hidden)
enc_out = torch.randn(batch, seq_len, hidden)
dec_hid = torch.randn(batch, hidden)

context, weights = attn(dec_hid, enc_out)
print(f" Encoder output: {enc_out.shape}")
print(f" Decoder hidden: {dec_hid.shape}")
print(f" Context vector: {context.shape}")
print(f" Attention weights: {weights.shape}")
print(f" Weights sum to 1: {weights.sum(dim=1)}")

# === OAI-11C: Bidirectional LSTM classifier ===
print("\n=== Bidirectional LSTM — Klasyfikacja sekwencji ===")

class BiLSTMClassifier(nn.Module):
 """Dwukierunkowy LSTM do klasyfikacji sekwencji."""
 def __init__(self, input_dim, hidden_dim, num_classes, n_layers=1):
 super().__init__()
 self.lstm = nn.LSTM(input_dim, hidden_dim, n_layers, 
 batch_first=True, bidirectional=True)
 self.classifier = nn.Linear(hidden_dim * 2, num_classes) # *2 bo bidirectional

 def forward(self, x):
 lstm_out, (h_n, _) = self.lstm(x)
 # Łączymy forward i backward hidden states
 h_forward = h_n[-2] # Ostatni forward
 h_backward = h_n[-1] # Ostatni backward
 combined = torch.cat([h_forward, h_backward], dim=1)
 return self.classifier(combined)

# Test
model_bi = BiLSTMClassifier(input_dim=1, hidden_dim=16, num_classes=3)
x_seq = torch.randn(4, 50, 1) # 4 sekwencje, 50 kroków, 1 cecha
out = model_bi(x_seq)
print(f" Input: {x_seq.shape} → Output: {out.shape}")
print(f" Parametry: {sum(p.numel() for p in model_bi.parameters()):,}")
print(f"\n Bidirectional LSTM 'widzi' sekwencję w obu kierunkach — lepsze dla klasyfikacji!")